# SAHA — TB Cough Screening Model Training Pipeline

**Scientifically valid, fully offline, real-world deployable TB cough screening.**

## Architecture
- **Input**: 16 kHz mono audio → Log-Mel Spectrogram (64 mel bins)
- **Model**: Lightweight 3-layer CNN classifier
- **Output**: Binary sigmoid → TB risk probability
- **Noise Robustness**: Trained with environmental noise augmentation
- **Deployment**: INT8 TFLite

## Safe Output Mapping
- p < 0.30 → **Low TB Risk**
- 0.30 ≤ p < 0.60 → **Moderate TB Risk**
- p ≥ 0.60 → **High TB Risk**
- Audio quality fails → **Inconclusive**
- Never outputs diagnosis

## Datasets
1. **Primary**: Sarcos TB cough dataset / TBScreen / Coswara
2. **Noise augmentation**: ESC-50 environmental sounds

---

⚠️ **Disclaimer**: AI Screening Tool. Not a medical diagnosis.

## 0. Environment Setup

In [ ]:
# ── Environment Setup ─────────────────────────────────────────────────────
!pip install -q tensorflow tensorflow-model-optimization scikit-learn matplotlib seaborn librosa soundfile pandas

import os
import json
import hashlib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_model_optimization as tfmot
import librosa
import librosa.display

from pathlib import Path
from collections import Counter
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve
)
from sklearn.utils.class_weight import compute_class_weight
from scipy.optimize import minimize_scalar

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

print(f'TensorFlow: {tf.__version__}')
print(f'Librosa: {librosa.__version__}')

# Reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Paths
BASE_DIR = Path('/kaggle/working')
OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

# Audio constants
SAMPLE_RATE = 16000
DURATION_SEC = 3
TOTAL_SAMPLES = SAMPLE_RATE * DURATION_SEC  # 48000
N_MELS = 64
N_FFT = 1024
HOP_LENGTH = 512
# Expected spectrogram shape: (N_MELS, ceil(TOTAL_SAMPLES/HOP_LENGTH)+1) = (64, 94)
SPEC_WIDTH = int(np.ceil(TOTAL_SAMPLES / HOP_LENGTH)) + 1  # 94 or 95
TARGET_SPEC_SHAPE = (N_MELS, 94)  # Fixed width

## 1. Dataset Assembly

### Dataset Strategy

TB cough datasets are rare. We search Kaggle for:
1. **Sarcos / TBScreen** — gold-standard TB-confirmed cough recordings
2. **Coswara** — large respiratory sound database (COVID/healthy/symptomatic)
3. **COUGHVID** — 25k crowd-sourced cough recordings

If no pure TB dataset is available, use respiratory symptomatic vs. healthy as a proxy.

### Important Limitation
If using proxy labels (symptomatic→positive, healthy→negative), the model detects
**respiratory abnormality in cough sounds**, not TB specifically. This must be
documented and disclosed.

In [ ]:
# ── 1A. Dataset Loading ───────────────────────────────────────────────────
#
# Adjust the DATA_DIR to your Kaggle dataset path.
# Supports multiple dataset structures:
#   a) Folder-based: positive/ and negative/ subdirs with .wav/.flac files
#   b) CSV-based: metadata.csv with columns [filename, label, patient_id]
#   c) Coswara-style: participant folders with cough recordings

# --- Configuration ---
# Try Sarcos TB dataset first, fallback to Coswara
DATA_DIR = Path('/kaggle/input')  # Adjust to actual dataset path

def load_audio_dataset(data_dir):
    """Auto-detect dataset structure and load audio file paths + labels.
    
    Returns: list of dicts with keys: path, label (0=negative, 1=positive), patient_id
    """
    records = []
    
    # --- Strategy 1: Folder-based (positive/negative dirs) ---
    for subdir in data_dir.rglob('*'):
        if not subdir.is_dir():
            continue
        name_lower = subdir.name.lower()
        label = None
        if any(kw in name_lower for kw in ['positive', 'tb_positive', 'tb', 'symptomatic', 'abnormal']):
            label = 1
        elif any(kw in name_lower for kw in ['negative', 'tb_negative', 'healthy', 'normal']):
            label = 0
        
        if label is not None:
            for audio_file in sorted(subdir.glob('*')):
                if audio_file.suffix.lower() in ('.wav', '.flac', '.mp3', '.ogg', '.webm'):
                    # Patient ID from filename prefix
                    patient_id = audio_file.stem.split('_')[0].split('-')[0]
                    records.append({
                        'path': str(audio_file),
                        'label': label,
                        'patient_id': f'{subdir.name}_{patient_id}',
                    })
    
    # --- Strategy 2: CSV metadata ---
    if not records:
        for csv_file in data_dir.rglob('*.csv'):
            try:
                meta = pd.read_csv(csv_file)
                # Look for common column patterns
                label_col = None
                file_col = None
                for col in meta.columns:
                    if any(kw in col.lower() for kw in ['label', 'class', 'diagnosis', 'status']):
                        label_col = col
                    if any(kw in col.lower() for kw in ['file', 'path', 'audio', 'filename']):
                        file_col = col
                
                if label_col and file_col:
                    for _, row in meta.iterrows():
                        label_val = str(row[label_col]).lower()
                        if any(kw in label_val for kw in ['positive', 'tb', 'abnormal', '1']):
                            label = 1
                        elif any(kw in label_val for kw in ['negative', 'healthy', 'normal', '0']):
                            label = 0
                        else:
                            continue
                        
                        audio_path = data_dir / row[file_col]
                        if not audio_path.exists():
                            audio_path = csv_file.parent / row[file_col]
                        if audio_path.exists():
                            pid = row.get('patient_id', row.get('participant_id', audio_path.stem.split('_')[0]))
                            records.append({
                                'path': str(audio_path),
                                'label': label,
                                'patient_id': str(pid),
                            })
            except Exception as e:
                print(f'  Skipping {csv_file.name}: {e}')
    
    return records

records = load_audio_dataset(DATA_DIR)
df = pd.DataFrame(records)

if len(df) == 0:
    print('⚠️ No audio records found. Check DATA_DIR path and dataset structure.')
    print(f'  DATA_DIR: {DATA_DIR}')
    print(f'  Contents: {list(DATA_DIR.iterdir())[:10]}')
else:
    print(f'Total audio samples: {len(df)}')
    print(f'Label distribution: {dict(df["label"].value_counts())}')
    print(f'Unique patients: {df["patient_id"].nunique()}')

In [ ]:
# ── 1B. Balance Dataset & Patient-Level Split ─────────────────────────────

# Balance classes by undersampling majority class
min_class_count = df['label'].value_counts().min()
df_balanced = df.groupby('label').apply(
    lambda x: x.sample(n=min(len(x), min_class_count * 2), random_state=SEED)
).reset_index(drop=True)

print(f'Balanced dataset: {len(df_balanced)} samples')
print(f'  Label distribution: {dict(df_balanced["label"].value_counts())}')

# --- Patient-level train/val/test split (70/15/15) ---
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_val_idx, test_idx = next(
    gss_test.split(df_balanced, df_balanced['label'], groups=df_balanced['patient_id'])
)
df_train_val = df_balanced.iloc[train_val_idx].reset_index(drop=True)
df_test = df_balanced.iloc[test_idx].reset_index(drop=True)

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.176, random_state=SEED)
train_idx, val_idx = next(
    gss_val.split(df_train_val, df_train_val['label'], groups=df_train_val['patient_id'])
)
df_train = df_train_val.iloc[train_idx].reset_index(drop=True)
df_val = df_train_val.iloc[val_idx].reset_index(drop=True)

# Verify no leakage
train_patients = set(df_train['patient_id'])
val_patients = set(df_val['patient_id'])
test_patients = set(df_test['patient_id'])
assert len(train_patients & val_patients) == 0, 'LEAKAGE: Train ∩ Val!'
assert len(train_patients & test_patients) == 0, 'LEAKAGE: Train ∩ Test!'
assert len(val_patients & test_patients) == 0, 'LEAKAGE: Val ∩ Test!'
print('✓ Zero data leakage confirmed')

for name, split_df in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    print(f'{name}: {len(split_df)} samples, {split_df["patient_id"].nunique()} patients')
    print(f'  {dict(split_df["label"].value_counts())}')

## 2. Feature Extraction — Log-Mel Spectrograms + Noise Augmentation

In [ ]:
# ── 2A. Audio Processing Functions ────────────────────────────────────────

def load_audio(path, sr=SAMPLE_RATE, duration=DURATION_SEC):
    """Load audio, resample to 16kHz, pad/truncate to fixed duration."""
    try:
        y, _ = librosa.load(path, sr=sr, mono=True, duration=duration + 0.5)
    except Exception as e:
        print(f'  Error loading {path}: {e}')
        return None
    
    target_len = sr * duration  # 48000
    if len(y) < target_len:
        # Pad with zeros
        y = np.pad(y, (0, target_len - len(y)), mode='constant')
    else:
        y = y[:target_len]
    
    return y

def extract_log_mel(y, sr=SAMPLE_RATE, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH):
    """Extract log-mel spectrogram.
    
    Returns: (n_mels, time_steps) float32 numpy array
    """
    mel_spec = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=n_mels, n_fft=n_fft,
        hop_length=hop_length, fmin=50, fmax=sr // 2
    )
    log_mel = librosa.power_to_db(mel_spec, ref=np.max)
    
    # Fix width to TARGET_SPEC_SHAPE[1]
    target_w = TARGET_SPEC_SHAPE[1]
    if log_mel.shape[1] < target_w:
        log_mel = np.pad(log_mel, ((0, 0), (0, target_w - log_mel.shape[1])), mode='constant', constant_values=-80)
    else:
        log_mel = log_mel[:, :target_w]
    
    return log_mel.astype(np.float32)

# Test on one sample
test_audio = load_audio(df_train['path'].iloc[0])
if test_audio is not None:
    test_spec = extract_log_mel(test_audio)
    print(f'Audio shape: {test_audio.shape}')
    print(f'Spectrogram shape: {test_spec.shape}')  # Expected: (64, 94)
    print(f'Spectrogram range: [{test_spec.min():.1f}, {test_spec.max():.1f}]')
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(test_audio)
    axes[0].set_title('Waveform', fontweight='bold')
    axes[0].set_xlabel('Sample')
    
    librosa.display.specshow(test_spec, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                            x_axis='time', y_axis='mel', ax=axes[1])
    axes[1].set_title('Log-Mel Spectrogram', fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 2B. Noise Augmentation ────────────────────────────────────────────────
#
# Add environmental noise at random SNR to improve generalization:
#   - Background chatter / babble
#   - Fan / AC noise
#   - Outdoor ambient noise
#
# If ESC-50 is available, use real noise. Otherwise, generate synthetic.

ESC50_DIR = Path('/kaggle/input/esc50')  # Adjust if using ESC-50

def load_noise_samples(noise_dir=None, n_samples=50):
    """Load noise samples from ESC-50 or generate synthetic."""
    noise_bank = []
    
    if noise_dir and noise_dir.exists():
        # Load real environmental noise from ESC-50
        # Categories: engine, rain, clock_tick, helicopter, chainsaw, etc.
        noise_files = list(noise_dir.rglob('*.wav'))[:n_samples]
        for nf in noise_files:
            try:
                n, _ = librosa.load(nf, sr=SAMPLE_RATE, mono=True, duration=DURATION_SEC)
                if len(n) < TOTAL_SAMPLES:
                    n = np.tile(n, int(np.ceil(TOTAL_SAMPLES / len(n))))[:TOTAL_SAMPLES]
                else:
                    n = n[:TOTAL_SAMPLES]
                noise_bank.append(n)
            except:
                pass
        print(f'Loaded {len(noise_bank)} real noise samples from ESC-50')
    
    # Generate synthetic noise if needed
    rng = np.random.RandomState(SEED)
    if len(noise_bank) < 10:
        # White noise
        for _ in range(10):
            noise_bank.append(rng.randn(TOTAL_SAMPLES).astype(np.float32) * 0.01)
        # Pink noise (1/f)
        for _ in range(10):
            white = rng.randn(TOTAL_SAMPLES).astype(np.float32)
            # Simple 1/f approximation via cumulative sum + normalization
            pink = np.cumsum(white)
            pink = pink / np.max(np.abs(pink)) * 0.01
            noise_bank.append(pink)
        # Low-frequency hum (fan noise)
        for freq in [50, 60, 100, 120]:
            t = np.linspace(0, DURATION_SEC, TOTAL_SAMPLES)
            hum = np.sin(2 * np.pi * freq * t).astype(np.float32) * 0.005
            noise_bank.append(hum)
        print(f'Generated {len(noise_bank)} synthetic noise samples')
    
    return noise_bank

noise_bank = load_noise_samples(ESC50_DIR)

def add_noise(audio, noise_bank, snr_db_range=(5, 20)):
    """Add noise at random SNR."""
    rng = np.random.RandomState()
    noise = noise_bank[rng.randint(len(noise_bank))]
    snr_db = rng.uniform(*snr_db_range)
    
    # Calculate scaling factor for desired SNR
    signal_power = np.mean(audio ** 2)
    noise_power = np.mean(noise ** 2)
    if noise_power < 1e-10:
        return audio
    
    target_noise_power = signal_power / (10 ** (snr_db / 10))
    scale = np.sqrt(target_noise_power / noise_power)
    
    return audio + noise * scale

def time_stretch(audio, rate_range=(0.9, 1.1)):
    """Time-stretch audio by random factor."""
    rate = np.random.uniform(*rate_range)
    stretched = librosa.effects.time_stretch(audio, rate=rate)
    if len(stretched) < TOTAL_SAMPLES:
        stretched = np.pad(stretched, (0, TOTAL_SAMPLES - len(stretched)))
    return stretched[:TOTAL_SAMPLES]

def pitch_shift(audio, sr=SAMPLE_RATE, semitone_range=(-2, 2)):
    """Pitch-shift audio by random semitones."""
    n_steps = np.random.uniform(*semitone_range)
    return librosa.effects.pitch_shift(audio, sr=sr, n_steps=n_steps)

In [ ]:
# ── 2C. Extract Features for All Splits ───────────────────────────────────

def extract_features(dataframe, augment=False, noise_bank=None, n_augments=2):
    """Extract log-mel spectrograms for all samples.
    
    Args:
        dataframe: DataFrame with 'path' and 'label' columns
        augment: Whether to apply augmentation
        noise_bank: Noise samples for augmentation
        n_augments: Number of augmented copies per sample
    
    Returns: (features, labels) numpy arrays
    """
    features = []
    labels = []
    
    for _, row in dataframe.iterrows():
        audio = load_audio(row['path'])
        if audio is None:
            continue
        
        # Original
        spec = extract_log_mel(audio)
        features.append(spec)
        labels.append(row['label'])
        
        # Augmented copies (training only)
        if augment and noise_bank:
            for _ in range(n_augments):
                aug_audio = audio.copy()
                
                # Random augmentation pipeline
                if np.random.random() < 0.5:
                    aug_audio = add_noise(aug_audio, noise_bank)
                if np.random.random() < 0.3:
                    aug_audio = time_stretch(aug_audio)
                if np.random.random() < 0.3:
                    aug_audio = pitch_shift(aug_audio)
                
                aug_spec = extract_log_mel(aug_audio)
                features.append(aug_spec)
                labels.append(row['label'])
    
    return np.array(features), np.array(labels)

print('Extracting training features (with augmentation)...')
X_train, y_train = extract_features(df_train, augment=True, noise_bank=noise_bank, n_augments=2)

print('Extracting validation features...')
X_val, y_val = extract_features(df_val)

print('Extracting test features...')
X_test, y_test = extract_features(df_test)

print(f'\nTrain: X={X_train.shape}, y={y_train.shape}, pos={y_train.sum()}')
print(f'Val:   X={X_val.shape}, y={y_val.shape}, pos={y_val.sum()}')
print(f'Test:  X={X_test.shape}, y={y_test.shape}, pos={y_test.sum()}')

In [ ]:
# ── 2D. Normalize Features ────────────────────────────────────────────────
# Per-channel (mel bin) z-score normalization using training statistics.

# Compute training mean and std per mel bin
train_mean = X_train.mean(axis=(0, 2), keepdims=True)  # (1, n_mels, 1)
train_std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6

# Normalize all splits
X_train = (X_train - train_mean) / train_std
X_val = (X_val - train_mean) / train_std
X_test = (X_test - train_mean) / train_std

# Add channel dimension for CNN: (N, H, W) → (N, H, W, 1)
X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print(f'Normalized shapes: train={X_train.shape}, val={X_val.shape}, test={X_test.shape}')
print(f'Train range: [{X_train.min():.2f}, {X_train.max():.2f}]')

# Save normalization stats for Dart inference
norm_stats = {
    'mean': train_mean.flatten().tolist(),
    'std': train_std.flatten().tolist(),
    'n_mels': N_MELS,
    'n_fft': N_FFT,
    'hop_length': HOP_LENGTH,
    'sample_rate': SAMPLE_RATE,
    'duration_sec': DURATION_SEC,
    'spec_width': TARGET_SPEC_SHAPE[1],
}
with open(OUTPUT_DIR / 'audio_norm_stats.json', 'w') as f:
    json.dump(norm_stats, f, indent=2)

print('Saved: audio_norm_stats.json')

## 3. Model Architecture — Lightweight CNN

In [ ]:
# ── 3. Model Architecture ─────────────────────────────────────────────────
#
# Lightweight 3-layer CNN on log-mel spectrograms.
# Proven effective for cough classification (Pahar et al. 2022).
# Much lighter than Wav2Vec2 — suitable for mobile deployment.

def build_cough_model(input_shape):
    """Build lightweight CNN for cough classification.
    
    Architecture:
        Conv2D(32) → BN → MaxPool → 
        Conv2D(64) → BN → MaxPool →
        Conv2D(128) → BN → GAP →
        Dropout(0.4) → Dense(64, ReLU, named 'embedding') →
        Dropout(0.3) → Dense(1, sigmoid)
    
    The Dense(64) layer is the embedding layer for potential OOD detection.
    """
    inputs = tf.keras.Input(shape=input_shape, name='spectrogram_input')
    
    # Block 1
    x = tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same', name='conv1')(inputs)
    x = tf.keras.layers.BatchNormalization(name='bn1')(x)
    x = tf.keras.layers.MaxPooling2D((2, 2), name='pool1')(x)
    
    # Block 2
    x = tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='conv2')(x)
    x = tf.keras.layers.BatchNormalization(name='bn2')(x)
    x = tf.keras.layers.MaxPooling2D((2, 2), name='pool2')(x)
    
    # Block 3
    x = tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same', name='conv3')(x)
    x = tf.keras.layers.BatchNormalization(name='bn3')(x)
    x = tf.keras.layers.GlobalAveragePooling2D(name='gap')(x)
    
    # Classification head
    x = tf.keras.layers.Dropout(0.4, name='dropout_1')(x)
    embedding = tf.keras.layers.Dense(64, activation='relu', name='embedding')(x)
    x = tf.keras.layers.Dropout(0.3, name='dropout_2')(embedding)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid', name='prediction')(x)
    
    model = tf.keras.Model(inputs, outputs, name='tb_cough_cnn')
    return model

input_shape = (N_MELS, TARGET_SPEC_SHAPE[1], 1)  # (64, 94, 1)
model = build_cough_model(input_shape)
model.summary()

print(f'\nTotal params: {model.count_params():,}')
print(f'Input shape: {input_shape}')

## 4. Training Protocol

In [ ]:
# ── 4. Training ───────────────────────────────────────────────────────────

# Class weights
cw_arr = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weights = {0: cw_arr[0], 1: cw_arr[1]}
print(f'Class weights: {class_weights}')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc', patience=10, mode='max',
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        str(OUTPUT_DIR / 'best_cough_model.keras'),
        monitor='val_auc', mode='max', save_best_only=True, verbose=1
    ),
]

print('\nTraining TB cough classifier...')
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# ── 4B. Training Curves ───────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history.history['loss'], label='Train'); axes[0].plot(history.history['val_loss'], label='Val')
axes[0].set_title('Loss', fontweight='bold'); axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train'); axes[1].plot(history.history['val_accuracy'], label='Val')
axes[1].set_title('Accuracy', fontweight='bold'); axes[1].legend()

axes[2].plot(history.history['auc'], label='Train'); axes[2].plot(history.history['val_auc'], label='Val')
axes[2].set_title('AUROC', fontweight='bold'); axes[2].legend()

for ax in axes: ax.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'tb_training_curves.png'), dpi=150)
plt.show()

## 5. Five-Fold Cross-Validation

In [ ]:
# ── 5. 5-Fold Stratified Group CV ─────────────────────────────────────────

cv_results = []
df_cv = pd.concat([df_train, df_val]).reset_index(drop=True)

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)

for fold, (tr_idx, vl_idx) in enumerate(sgkf.split(df_cv, df_cv['label'], groups=df_cv['patient_id'])):
    print(f'\n══════ Fold {fold+1}/5 ══════')
    
    fold_train_df = df_cv.iloc[tr_idx]
    fold_val_df = df_cv.iloc[vl_idx]
    assert len(set(fold_train_df['patient_id']) & set(fold_val_df['patient_id'])) == 0
    
    # Extract features
    fold_X_train, fold_y_train = extract_features(fold_train_df, augment=True, noise_bank=noise_bank, n_augments=1)
    fold_X_val, fold_y_val = extract_features(fold_val_df)
    
    # Normalize
    fold_mean = fold_X_train.mean(axis=(0, 2), keepdims=True)
    fold_std = fold_X_train.std(axis=(0, 2), keepdims=True) + 1e-6
    fold_X_train = ((fold_X_train - fold_mean) / fold_std)[..., np.newaxis]
    fold_X_val = ((fold_X_val - fold_mean) / fold_std)[..., np.newaxis]
    
    # Build & train
    fold_model = build_cough_model(input_shape)
    fold_cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=fold_y_train)
    fold_model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='binary_crossentropy',
        metrics=[tf.keras.metrics.AUC(name='auc')]
    )
    fold_model.fit(
        fold_X_train, fold_y_train,
        validation_data=(fold_X_val, fold_y_val),
        epochs=20, batch_size=32,
        class_weight={0: fold_cw[0], 1: fold_cw[1]},
        callbacks=[tf.keras.callbacks.EarlyStopping('val_auc', patience=5, mode='max', restore_best_weights=True)],
        verbose=0
    )
    
    # Evaluate
    y_pred_probs = fold_model.predict(fold_X_val, verbose=0).flatten()
    y_pred = (y_pred_probs >= 0.5).astype(int)
    
    try:
        auroc = roc_auc_score(fold_y_val, y_pred_probs)
    except ValueError:
        auroc = 0.0
    
    cm = confusion_matrix(fold_y_val, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    fold_metrics = {'fold': fold+1, 'auroc': auroc, 'sensitivity': sens, 'specificity': spec}
    cv_results.append(fold_metrics)
    print(f'  AUROC={auroc:.4f}, Sens={sens:.4f}, Spec={spec:.4f}')
    
    del fold_model
    tf.keras.backend.clear_session()

cv_df = pd.DataFrame(cv_results)
print('\n══════ 5-Fold CV Summary ══════')
for col in ['auroc', 'sensitivity', 'specificity']:
    vals = cv_df[col].values
    print(f'  {col}: {vals.mean():.4f} ± {vals.std():.4f}')

cv_df.to_csv(str(OUTPUT_DIR / 'tb_cv_results.csv'), index=False)

## 6. Final Model Evaluation on Held-Out Test Set

In [ ]:
# ── 6A. Test Set Evaluation ───────────────────────────────────────────────

best_model = tf.keras.models.load_model(str(OUTPUT_DIR / 'best_cough_model.keras'))

y_test_probs = best_model.predict(X_test, verbose=0).flatten()
y_test_pred = (y_test_probs >= 0.5).astype(int)

# AUROC
test_auroc = roc_auc_score(y_test, y_test_probs)
print(f'Test AUROC: {test_auroc:.4f}')

# Confusion matrix
cm = confusion_matrix(y_test, y_test_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f'Sensitivity: {sensitivity:.4f}')
print(f'Specificity: {specificity:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_test_pred, target_names=['Negative', 'Positive'], digits=4))

# ── Performance at multiple thresholds ─────────────────────────────────────
print('\n=== Performance at Different Thresholds ===')
for threshold in [0.30, 0.40, 0.50, 0.60, 0.70]:
    preds = (y_test_probs >= threshold).astype(int)
    cm_t = confusion_matrix(y_test, preds, labels=[0, 1])
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    sens_t = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0
    spec_t = tn_t / (tn_t + fp_t) if (tn_t + fp_t) > 0 else 0
    print(f'  Threshold={threshold:.2f}: Sens={sens_t:.4f}, Spec={spec_t:.4f}')

In [ ]:
# ── 6B. Visualizations ────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_test_probs)
axes[0].plot(fpr, tpr, 'b-', lw=2, label=f'AUC={test_auroc:.3f}')
axes[0].plot([0,1], [0,1], 'k--', alpha=0.5)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve', fontweight='bold'); axes[0].legend()

# Confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'], ax=axes[1])
axes[1].set_title('Confusion Matrix', fontweight='bold')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')

# Score distribution
axes[2].hist(y_test_probs[y_test == 0], bins=30, alpha=0.6, label='Negative', color='green')
axes[2].hist(y_test_probs[y_test == 1], bins=30, alpha=0.6, label='Positive', color='red')
axes[2].axvline(0.3, color='orange', ls='--', label='Low/Mod threshold')
axes[2].axvline(0.6, color='red', ls='--', label='Mod/High threshold')
axes[2].set_xlabel('TB Probability'); axes[2].set_ylabel('Count')
axes[2].set_title('Score Distribution', fontweight='bold'); axes[2].legend()

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'tb_evaluation.png'), dpi=150)
plt.show()

In [ ]:
# ── 6C. Evaluate on Noisy Test Samples ────────────────────────────────────
#
# Simulate real-world conditions: add noise to test set and re-evaluate.

print('=== Noise Robustness Evaluation ===')
for snr in [20, 15, 10, 5]:
    # Re-extract test features with noise
    noisy_X = []
    for spec in X_test:
        # Inverse the normalization, add noise to raw audio-approximation via spec perturbation
        noise_spec = spec + np.random.randn(*spec.shape).astype(np.float32) * (0.1 * (20 / snr))
        noisy_X.append(noise_spec)
    noisy_X = np.array(noisy_X)
    
    noisy_probs = best_model.predict(noisy_X, verbose=0).flatten()
    noisy_auroc = roc_auc_score(y_test, noisy_probs)
    noisy_pred = (noisy_probs >= 0.5).astype(int)
    cm_n = confusion_matrix(y_test, noisy_pred, labels=[0, 1])
    tn_n, fp_n, fn_n, tp_n = cm_n.ravel()
    noisy_sens = tp_n / (tp_n + fn_n) if (tp_n + fn_n) > 0 else 0
    
    print(f'  SNR={snr}dB: AUROC={noisy_auroc:.4f}, Sens={noisy_sens:.4f}')

## 7. Quantization & TFLite Export

In [ ]:
# ── 7A. Quantization-Aware Training ───────────────────────────────────────

print('Applying QAT to best model...')
qat_model = tfmot.quantization.keras.quantize_model(best_model)

qat_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=[tf.keras.metrics.AUC(name='auc')]
)

qat_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5, batch_size=32,
    class_weight=class_weights,
    verbose=1
)

# Evaluate QAT
qat_probs = qat_model.predict(X_test, verbose=0).flatten()
qat_auroc = roc_auc_score(y_test, qat_probs)
print(f'QAT AUROC: {qat_auroc:.4f} (vs FP32: {test_auroc:.4f})')
print(f'Drop: {(test_auroc - qat_auroc)*100:.2f}%')

In [ ]:
# ── 7B. Convert to TFLite INT8 ───────────────────────────────────────────

def representative_dataset():
    """Generator for quantization calibration."""
    for i in range(min(200, len(X_train))):
        yield [X_train[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(qat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.float32  # Keep float input for spectrogram
converter.inference_output_type = tf.float32

print('Converting to TFLite...')
tflite_model = converter.convert()

tflite_path = OUTPUT_DIR / 'tb_cough_classifier.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

model_size_kb = os.path.getsize(tflite_path) / 1024
sha256 = hashlib.sha256(tflite_model).hexdigest()

print(f'TFLite saved: {tflite_path}')
print(f'Size: {model_size_kb:.1f} KB')
print(f'SHA-256: {sha256}')

In [ ]:
# ── 7C. Validate TFLite Accuracy ─────────────────────────────────────────

interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f'Input: {input_details[0]["shape"]}, dtype={input_details[0]["dtype"]}')
print(f'Output: {output_details[0]["shape"]}, dtype={output_details[0]["dtype"]}')

# Run inference
tflite_preds = []
for i in range(len(X_test)):
    inp = X_test[i:i+1].astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], inp)
    interpreter.invoke()
    out = interpreter.get_tensor(output_details[0]['index'])
    tflite_preds.append(out[0][0])

tflite_preds = np.array(tflite_preds)
tflite_auroc = roc_auc_score(y_test, tflite_preds)

print(f'\n=== TFLite Validation ===')
print(f'FP32 AUROC:   {test_auroc:.4f}')
print(f'TFLite AUROC: {tflite_auroc:.4f}')
print(f'Drop: {(test_auroc - tflite_auroc)*100:.2f}%')

if abs(test_auroc - tflite_auroc) < 0.03:
    print('✓ Quantization drop < 3% — PASS')
else:
    print('✗ WARNING: Quantization drop > 3%')

# Benchmark
import time
times = []
sample = np.random.randn(1, *input_shape).astype(np.float32)
for _ in range(50):
    t0 = time.perf_counter()
    interpreter.set_tensor(input_details[0]['index'], sample)
    interpreter.invoke()
    times.append(time.perf_counter() - t0)

print(f'\nInference benchmark (50 runs): Mean={np.mean(times)*1000:.1f}ms, P95={np.percentile(times,95)*1000:.1f}ms')

## 8. Export Summary

In [ ]:
# ── 8. Final Summary ──────────────────────────────────────────────────────

summary = {
    'model_name': 'tb_cough_classifier',
    'version': '2.0.0',
    'architecture': '3-layer CNN on log-mel spectrograms',
    'input_shape': list(input_shape),
    'output': 'sigmoid (TB risk probability)',
    'quantization': 'INT8 (QAT)',
    'model_size_kb': round(model_size_kb, 1),
    'sha256': sha256,
    'audio_params': {
        'sample_rate': SAMPLE_RATE,
        'duration_sec': DURATION_SEC,
        'n_mels': N_MELS,
        'n_fft': N_FFT,
        'hop_length': HOP_LENGTH,
    },
    'safe_output_thresholds': {
        'low_risk_below': 0.30,
        'moderate_risk_below': 0.60,
        'high_risk_above': 0.60,
    },
    'metrics': {
        'test_auroc': round(float(test_auroc), 4),
        'test_sensitivity': round(float(sensitivity), 4),
        'test_specificity': round(float(specificity), 4),
        'tflite_auroc': round(float(tflite_auroc), 4),
        'quantization_drop_pct': round(float(test_auroc - tflite_auroc) * 100, 2),
    },
    'cv_auroc_mean': round(float(cv_df['auroc'].mean()), 4),
    'cv_auroc_std': round(float(cv_df['auroc'].std()), 4),
    'disclaimer': 'AI Screening Tool. Not a medical diagnosis.',
}

with open(OUTPUT_DIR / 'tb_model_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

print('\n=== Output Files ===')
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {f.name}: {f.stat().st_size/1024:.1f} KB')

print('\n✓ TB cough training pipeline complete.')
print('  Copy tb_cough_classifier.tflite → assets/models/')
print('  Copy audio_norm_stats.json values → tb_audio_classifier.dart')

## 8B. TensorFlow.js Export (for SAHA Web)

Exports the trained Keras model to TF.js LayersModel format for browser-based
inference. The output `model.json` + weight shard `.bin` files should be copied
to `web/models/tb_cough/` in the Flutter project.

In [ ]:
# ── TF.js Export ──────────────────────────────────────────────────────────
!pip install -q tensorflowjs
import tensorflowjs as tfjs

tfjs_output = OUTPUT_DIR / 'tfjs_tb_cough'
tfjs.converters.save_keras_model(best_model, str(tfjs_output))

print(f'\n=== TF.js Model Files ===')
for f in sorted(tfjs_output.iterdir()):
    print(f'  {f.name}: {f.stat().st_size/1024:.1f} KB')

tfjs_total = sum(f.stat().st_size for f in tfjs_output.iterdir())
print(f'  Total: {tfjs_total/1024:.1f} KB')
print('\n✓ TF.js export complete.')
print('  Copy contents of outputs/tfjs_tb_cough/ → web/models/tb_cough/')

## Kaggle Execution Instructions

1. **Create a new Kaggle Notebook** at https://www.kaggle.com/code
2. **Add Datasets** (right panel → "+ Add Data"):
   - Any TB cough dataset (search: "tb cough", "coughvid", "coswara")
   - `esc-50` (for noise augmentation, optional)
3. **Enable GPU** (Settings → Accelerator → GPU T4 x2)
4. **Upload** this notebook or paste all cells
5. **Click "Run All"** — takes ~20-30 minutes
6. **Download outputs** from `/kaggle/working/outputs/`:
   - `tb_cough_classifier.tflite` → `assets/models/`
   - `tfjs_tb_cough/` folder contents → `web/models/tb_cough/`
   - `audio_norm_stats.json` → update `_normMean` and `_normStd` in `tb_audio_classifier.dart`
   - `tb_model_summary.json` → note the `sha256` value
7. **Update Dart code**:
   - Replace `_normMean` and `_normStd` arrays in `tb_audio_classifier.dart`
   - Set SHA-256 in `model_governance.dart` model registry

## Testing Checklist for Real-World Validation

| # | Test | Expected Result | Status |
|---|------|-----------------|--------|
| 1 | Record TB-like cough | High TB Risk (p ≥ 0.60) | ☐ |
| 2 | Record healthy cough | Low TB Risk (p < 0.30) | ☐ |
| 3 | Record in noisy environment | Should still detect correctly | ☐ |
| 4 | Record silence | Audio quality gate → **Inconclusive** | ☐ |
| 5 | Record speech only | Cough detector fails → **Inconclusive** | ☐ |
| 6 | Record < 3 seconds | Duration check → **Retake** | ☐ |
| 7 | Record clipped audio | Clipping check → **Retake** | ☐ |
| 8 | Inference time < 2s | Benchmark on device | ☐ |
| 9 | Model size < 1 MB | Check .tflite file | ☐ |
| 10 | AUROC ≥ 0.75 | Check test evaluation | ☐ |

---

⚠️ **This is a screening tool, not a diagnostic device.**

All results MUST be confirmed by qualified healthcare professionals.